# 06 — Segmentasi Pelanggan: K-Means (Bab 2.4b)Notebook ini mengimplementasikan **segmentasi pelanggan** menggunakan algoritma **K-Means Clustering** dari Spark MLlib.

## 6.1 Inisialisasi & Load Data

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum as spark_sum

spark = SparkSession.builder \
    .appName("06_KMeans_Segmentation") \
    .master("local[*]") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

df = spark.read.parquet("/output/retail_parquet")
print(f"✅ Loaded {df.count()} rows")

✅ Loaded 1000 rows


## 6.2 Feature EngineeringBuat fitur per pelanggan: frequency, monetary, avg_quantity

In [10]:
# Agregasi fitur per pelanggan
df_customer = df.groupBy("Customer_ID").agg(
    count("*").alias("frequency"),
    avg("Total_Amount").alias("avg_monetary"),
    avg("Quantity").alias("avg_quantity"),
    spark_sum("Total_Amount").alias("total_spending")
)

print(f"✅ {df_customer.count()} pelanggan unik")
df_customer.show(10)
df_customer.describe().show()

✅ 1000 pelanggan unik
+-----------+---------+------------+------------+--------------+
|Customer_ID|frequency|avg_monetary|avg_quantity|total_spending|
+-----------+---------+------------+------------+--------------+
|    CUST580|        1|      1500.0|         3.0|          1500|
|    CUST748|        1|       150.0|         3.0|           150|
|    CUST690|        1|       900.0|         3.0|           900|
|    CUST729|        1|      1200.0|         4.0|          1200|
|    CUST359|        1|        50.0|         1.0|            50|
|    CUST438|        1|        30.0|         1.0|            30|
|    CUST696|        1|       200.0|         4.0|           200|
|    CUST103|        1|        25.0|         1.0|            25|
|    CUST316|        1|        50.0|         2.0|            50|
|    CUST844|        1|       150.0|         3.0|           150|
+-----------+---------+------------+------------+--------------+
only showing top 10 rows

+-------+-----------+---------+-----------

## 6.3 Persiapan Fitur untuk MLlib

In [11]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Assemble fitur ke vector
assembler = VectorAssembler(
    inputCols=["frequency", "avg_monetary", "avg_quantity"],
    outputCol="features_raw"
)
df_assembled = assembler.transform(df_customer)

# Standarisasi fitur (penting untuk K-Means!)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

print("✅ Fitur di-assemble dan di-scale")
df_scaled.select("Customer_ID", "features").show(5, truncate=False)

✅ Fitur di-assemble dan di-scale
+-----------+---------------------------------------------+
|Customer_ID|features                                     |
+-----------+---------------------------------------------+
|CUST580    |[0.0,1.8642935990654013,0.4290502922402973]  |
|CUST748    |[0.0,-0.5464308824846851,0.4290502922402973] |
|CUST690    |[0.0,0.7928604961542517,0.4290502922402973]  |
|CUST729    |[0.0,1.3285770476098266,1.3118698235989292]  |
|CUST359    |[0.0,-0.7250030663032101,-1.3365887704769668]|
+-----------+---------------------------------------------+
only showing top 5 rows



## 6.4 Menentukan Jumlah Cluster Optimal (Elbow Method)

In [12]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Test k = 2 sampai 8
costs = []
silhouettes = []
evaluator = ClusteringEvaluator(featuresCol="features", metricName="silhouette")

print("=== ELBOW METHOD ===")
print(f"{'k':>3} | {'Cost (WSSSE)':>15} | {'Silhouette':>10}")
print("-" * 40)

for k in range(2, 9):
    kmeans = KMeans(k=k, seed=42, featuresCol="features", predictionCol="prediction")
    model = kmeans.fit(df_scaled)
    cost = model.summary.trainingCost
    
    predictions = model.transform(df_scaled)
    silhouette = evaluator.evaluate(predictions)
    
    costs.append(cost)
    silhouettes.append(silhouette)
    print(f"{k:>3} | {cost:>15.2f} | {silhouette:>10.4f}")

print("\n📌 Pilih k di mana cost mulai landai (elbow) dan silhouette tinggi")

=== ELBOW METHOD ===
  k |    Cost (WSSSE) | Silhouette
----------------------------------------
  2 |         1054.30 |     0.6578
  3 |          438.54 |     0.7564
  4 |          314.25 |     0.7606
  5 |          261.01 |     0.6512
  6 |          205.39 |     0.8094
  7 |          124.83 |     0.7557
  8 |           98.74 |     0.7619

📌 Pilih k di mana cost mulai landai (elbow) dan silhouette tinggi


## 6.5 Train K-Means (k=3)

In [13]:
# Final model dengan k=3
kmeans_final = KMeans(
    k=3, 
    seed=42, 
    featuresCol="features", 
    predictionCol="segment"
)
model_final = kmeans_final.fit(df_scaled)

# Transform
df_segmented = model_final.transform(df_scaled)
print("✅ K-Means training selesai (k=3)")
print(f"   WSSSE (Cost): {model_final.summary.trainingCost:.2f}")

✅ K-Means training selesai (k=3)
   WSSSE (Cost): 438.54


## 6.6 Analisis Hasil Segmentasi

In [14]:
print("=== PROFIL SETIAP SEGMEN ===\n")

df_segmented.groupBy("segment").agg(
    count("*").alias("jumlah_pelanggan"),
    avg("avg_monetary").alias("avg_spending"),
    avg("frequency").alias("avg_frequency"),
    avg("avg_quantity").alias("avg_quantity"),
    avg("total_spending").alias("avg_total_spending")
).orderBy("segment").show()

# Label segmen
print("📌 Interpretasi Segmen:")
print("   Segment dengan avg_spending tertinggi → High Value")
print("   Segment dengan avg_spending sedang    → Medium Value")  
print("   Segment dengan avg_spending terendah  → Low Value")

=== PROFIL SETIAP SEGMEN ===

+-------+----------------+------------------+-------------+------------------+------------------+
|segment|jumlah_pelanggan|      avg_spending|avg_frequency|      avg_quantity|avg_total_spending|
+-------+----------------+------------------+-------------+------------------+------------------+
|      0|             264|1297.7272727272727|          1.0|3.2045454545454546|1297.7272727272727|
|      1|             447|171.93512304250558|          1.0|1.4340044742729305|171.93512304250558|
|      2|             289|126.45328719723183|          1.0|3.5536332179930796|126.45328719723183|
+-------+----------------+------------------+-------------+------------------+------------------+

📌 Interpretasi Segmen:
   Segment dengan avg_spending tertinggi → High Value
   Segment dengan avg_spending sedang    → Medium Value
   Segment dengan avg_spending terendah  → Low Value


## 6.7 Distribusi Segmen

In [15]:
print("=== DISTRIBUSI SEGMEN ===\n")
total = df_segmented.count()
for row in df_segmented.groupBy("segment").count().orderBy("segment").collect():
    pct = row["count"] / total * 100
    print(f"   Segment {row['segment']}: {row['count']} pelanggan ({pct:.1f}%)")

=== DISTRIBUSI SEGMEN ===

   Segment 0: 264 pelanggan (26.4%)
   Segment 1: 447 pelanggan (44.7%)
   Segment 2: 289 pelanggan (28.9%)


## 6.8 Simpan Hasil Segmentasi

In [16]:
# Simpan ke Parquet
OUTPUT_PATH = "/output/customer_segments"

df_segmented.select(
    "Customer_ID", "frequency", "avg_monetary", 
    "avg_quantity", "total_spending", "segment"
).write.mode("overwrite").parquet(OUTPUT_PATH)

print(f"✅ Hasil segmentasi disimpan ke: {OUTPUT_PATH}")

✅ Hasil segmentasi disimpan ke: /output/customer_segments
